# VEILINK 六轴机械臂
# Xbox 手柄操控机械臂

本 Notebook 按“腕部中心定位 + 独立腕部控制”的方式操控七台 STS3215：

- J1～J3 协同移动 J4 附近的腕部中心 `O4`；
- J4、J5 控制夹爪相对腕部的指向；
- J6 控制夹爪绕末端轴旋转；
- ID7 控制夹爪张开和闭合。

本 Notebook 已内置手柄控制所需的运动学核心，包括最终 DH 参数读取、正运动学、关节限位、编码器换算和 `O4` 位置雅可比，不依赖外部运动学 Python 文件。

> **本 Notebook 会控制真实硬件。首次运行时，硬件连接与主循环均默认关闭。必须按章节顺序完成离线测试，并明确修改确认开关后才会访问串口。**

## 0. 最终控制映射

| Xbox 输入 | 功能 |
| --- | --- |
| 左摇杆 | 移动腕部中心 O4 前后、左右，由 J1～J3 协同实现 |
| 右摇杆上下 / 左右 | 背面视角下对应 O4 上下 / J4 滚转；其他视角按 90°旋转 |
| RB / LB | 分别控制 J5 正转 / 反转；同时按下时停止 J5 |
| 方向键左右 | 控制 J6 正反向旋转 |
| LT | 夹爪张开 |
| RT | 夹爪闭合 |
| X / Y | 速度提高 / 降低一档：20%、40%、60%、80%、100% |
| 方向键上 / 下 | 在背面、右侧、正面、左侧四个视角间循环切换 |
| A 短按 | 进入/退出可控制状态；退出后仍保持力矩；按下即震动 1 秒 |
| A 长按 2 秒 | 退出控制循环并保持当前位置和力矩 |
| B | 锁存紧急卸力、结束控制循环并震动 1 秒 |

短按 `A` 决定系统是否接受控制；进入可控制状态后，摇杆、方向键、扳机和 J5 按键直接产生运动。四个视角会把左右摇杆指令按 90°递进旋转；J5、J6 的按键正反转定义不变。长按 A 2 秒用于安全退出循环并保持力矩。

### 必须理解的腕部限制

J4、J5 只能改变夹爪指向，使 TCP 沿以腕部为中心的球面运动，不能产生任意平面直线位移。J5 接近 `0°` 时，J4 对 TCP 位置几乎没有作用；程序会显示腕部奇异提示，但仍允许低速调整 J4。

## 1. 实机安全要求

1. 选择 `Python (lerobot)` kernel，关闭其他占用串口或手柄的程序。
2. 机械臂必须固定在稳定底座上，首次运行不要安装负载。
3. J2、J3 卸力时机械臂可能下坠；B 键卸力前后都要确保机械臂有人托住或有支架。
4. 第一次只使用低速配置，每次推动摇杆不超过 1 秒，随时准备按 B 和切断外部电源。
5. J1、J4、J6 虽是单圈关节，本程序仍将其限制在 `±170°`，为线缆留下余量；不要绕过该限制连续旋转。
6. 软件 B 键不能替代实体急停。程序卡死、USB 断开或电源故障时，软件命令可能无法送达。
7. 出现异常运动、异响、发热、通信错误、结构干涉或线缆拉紧时，立即按 B 并切断舵机电源。

## 2. 加载环境、校准参数与内置运动学核心

本节直接定义只依赖 NumPy 的运动学核心。它从校准 JSON 读取最终标准 DH 参数和实测软限位，提供正运动学、关节角/编码器换算及硬件就绪检查；手柄控制不需要 SciPy 或外部运动学模块。

内置核心不是另一套参数，而是从同一份 `机械臂校准参数.json` 读取最终标准 DH 表，并实现本控制器实际需要的部分：

```text
θ = q + [0°, 90°, 90°, 180°, 0°, 0°]
^(i-1)Ti = Rz(θi) · Tz(di) · Tx(ai) · Rx(αi)
^0Ti = ^0T1 · ^1T2 · ... · ^(i-1)Ti
```

它还复用同一校准关系把物理角转换为编码器值。内置核心不包含本控制方式不需要的完整 TCP 位姿 IK，因此不会改变后续“J1～J3 定位 O4、J4～J6 独立控制”的算法。

In [ ]:
import html
import json
import os
import sys
import time
from importlib.metadata import version
from pathlib import Path

os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")

import numpy as np
import ipywidgets as widgets
from IPython.display import display

WORK_DIR = Path.cwd().resolve()
if WORK_DIR.name == "Code":
    CODE_DIR = WORK_DIR
elif (WORK_DIR / "Code").is_dir():
    CODE_DIR = WORK_DIR / "Code"
else:
    raise FileNotFoundError("请从项目根目录或‘Code’目录运行本 Notebook。")

PARAM_PATH = CODE_DIR / "机械臂校准参数.json"
params = json.loads(PARAM_PATH.read_text(encoding="utf-8"))

if params.get("schema_version") != 1:
    raise RuntimeError("不支持的校准参数 schema_version。")
if not params.get("calibrated"):
    raise RuntimeError("calibrated=false，禁止进入手柄控制。请先完成舵机校准。")
if params.get("kinematics_revision") != "2026-08-09-dh-v2":
    raise RuntimeError("运动学版本不是 2026-08-09-dh-v2。")
if not params.get("hardware", {}).get("gripper_enabled", False):
    raise RuntimeError("校准文件未启用夹爪，但本程序要求七台电机。")

JOINTS = ["J1", "J2", "J3", "J4", "J5", "J6"]
MOTOR_NAMES = JOINTS + ["gripper"]
missing_directions = [
    name for name in MOTOR_NAMES
    if not params["joints"][name].get("direction_verified", False)
]
if missing_directions:
    raise RuntimeError(f"以下舵机方向尚未确认：{missing_directions}")

class NotebookKinematicsCore:
    '''手柄控制专用的 NumPy 标准 DH、关节范围与编码器换算核心。'''

    def __init__(self, calibration):
        self.calibration = calibration
        rows = calibration.get("dh_parameters", [])
        if [row.get("joint") for row in rows] != JOINTS:
            raise RuntimeError("校准文件 dh_parameters 的关节顺序不是 J1～J6。")
        self.DH_D_MM = np.array([row["d_mm"] for row in rows], dtype=float)
        self.DH_A_MM = np.array([row["a_mm"] for row in rows], dtype=float)
        self.DH_ALPHA_DEG = np.array([row["alpha_deg"] for row in rows], dtype=float)
        self.DH_THETA_OFFSET_DEG = np.array(
            [row["theta_offset_deg"] for row in rows], dtype=float
        )
        published_min = np.array([row["design_min_deg"] for row in rows], dtype=float)
        published_max = np.array([row["design_max_deg"] for row in rows], dtype=float)
        self.Q_MIN_DEG = published_min.copy()
        self.Q_MAX_DEG = published_max.copy()
        full_turn = (published_max - published_min) >= 360.0
        self.Q_MIN_DEG[full_turn] = -180.0
        self.Q_MAX_DEG[full_turn] = 180.0
        self.ENCODER_RESOLUTION = int(
            calibration.get("hardware", {}).get("encoder_resolution", 4096)
        )
        for index, name in enumerate(JOINTS):
            item = calibration["joints"][name]
            if not item.get("continuous", False) and item.get("soft_min_raw") is not None:
                endpoints = [
                    self.calibration_angle_deg(item, item["soft_min_raw"]),
                    self.calibration_angle_deg(item, item["soft_max_raw"]),
                ]
                self.Q_MIN_DEG[index] = max(self.Q_MIN_DEG[index], min(endpoints))
                self.Q_MAX_DEG[index] = min(self.Q_MAX_DEG[index], max(endpoints))
        if np.any(self.Q_MIN_DEG >= self.Q_MAX_DEG):
            raise RuntimeError("设计范围与实测软限位没有有效交集。")
        self.HARDWARE_READY = bool(
            calibration.get("calibrated")
            and all(calibration["joints"][name].get("direction_verified", False) for name in JOINTS)
        )
    def wrapped_tick_delta(self, value, reference):
        half = self.ENCODER_RESOLUTION // 2
        return ((int(value) - int(reference) + half) % self.ENCODER_RESOLUTION) - half

    def calibration_angle_deg(self, item, raw_value):
        delta = self.wrapped_tick_delta(raw_value, item["zero_position_raw"])
        return int(item.get("direction", 1)) * delta * 360.0 / self.ENCODER_RESOLUTION

    @staticmethod
    def standard_dh_matrix(theta_rad, d_mm, a_mm, alpha_rad):
        ct, st = np.cos(theta_rad), np.sin(theta_rad)
        ca, sa = np.cos(alpha_rad), np.sin(alpha_rad)
        return np.array([
            [ct, -st * ca, st * sa, a_mm * ct],
            [st, ct * ca, -ct * sa, a_mm * st],
            [0.0, sa, ca, d_mm],
            [0.0, 0.0, 0.0, 1.0],
        ], dtype=float)

    def forward_kinematics(self, q_deg, return_all=False):
        q = np.asarray(q_deg, dtype=float)
        if q.size != 6 or not np.all(np.isfinite(q)):
            raise ValueError("q_deg 必须包含 6 个有限数。")
        theta = np.deg2rad(q.reshape(6) + self.DH_THETA_OFFSET_DEG)
        alpha = np.deg2rad(self.DH_ALPHA_DEG)
        transform = np.eye(4)
        frames = [transform.copy()]
        for index in range(6):
            transform = transform @ self.standard_dh_matrix(
                theta[index], self.DH_D_MM[index], self.DH_A_MM[index], alpha[index]
            )
            frames.append(transform.copy())
        return (transform, frames) if return_all else transform

    def joint_deg_to_servo_raw(self, q_deg):
        q = np.asarray(q_deg, dtype=float)
        if q.size != 6 or not np.all(np.isfinite(q)):
            raise ValueError("q_deg 必须包含 6 个有限数。")
        targets = {}
        for name, angle in zip(JOINTS, q.reshape(6)):
            item = self.calibration["joints"][name]
            direction = int(item.get("direction", 1))
            raw_unwrapped = int(item["zero_position_raw"]) + round(
                angle * self.ENCODER_RESOLUTION / (360.0 * direction)
            )
            if item.get("continuous", False):
                raw = raw_unwrapped % self.ENCODER_RESOLUTION
            else:
                raw = raw_unwrapped
                low, high = int(item["soft_min_raw"]), int(item["soft_max_raw"])
                if not low <= raw <= high:
                    raise ValueError(f"{name} raw={raw} 超出实测软限位 [{low}, {high}]。")
            targets[name] = int(raw)
        return targets

kin = NotebookKinematicsCore(params)

print("Python:", Path(sys.executable))
print("LeRobot:", version("lerobot"))
print("运动学核心：Notebook 内置 NumPy 标准 DH 实现")
print("校准文件：", PARAM_PATH)
print("hardware_ready =", kin.HARDWARE_READY)

## 3. 控制参数

所有参数先使用保守值。模型中 `+z0` 指向地面，因此“右摇杆向上”默认映射到负 `z0`。`BASE_X_FORWARD_SIGN` 和 `BASE_Y_RIGHT_SIGN` 取决于机械臂底座相对操作者的朝向，必须先在离线状态和短脉冲实机测试中核对；如果方向相反，只修改对应符号，不要修改 DH 参数或舵机方向。

In [ ]:
# 串口：默认读取校准记录；COM 号变化时只覆盖这里。
PORT = params["hardware"].get("port_used") or "COM3"

# 主循环和输入整形。
LOOP_HZ = 30.0
STICK_DEADZONE = 0.15
TRIGGER_DEADZONE = 0.08
STICK_CUBIC_MIX = 0.65

# 第 1 档（显示 20%）的保守速度基准；第 5 档为下列数值的 5 倍。
O4_SPEED_PER_LEVEL_MM_S = 15.0
JOINT_SPEED_PER_LEVEL_DEG_S = 10.0  # J1～J6 共用同一关节角速度基准
J123_SPEED_PER_LEVEL_DEG_S = JOINT_SPEED_PER_LEVEL_DEG_S
J4_SPEED_PER_LEVEL_DEG_S = JOINT_SPEED_PER_LEVEL_DEG_S
J5_SPEED_PER_LEVEL_DEG_S = JOINT_SPEED_PER_LEVEL_DEG_S
J6_SPEED_PER_LEVEL_DEG_S = JOINT_SPEED_PER_LEVEL_DEG_S
MAX_J6_TARGET_LEAD_DEG = 5.0  # J6 目标最多领先实际反馈 5°
GRIPPER_SPEED_PERCENT_S = 15.0

# X/Y 切换五档；内部倍率 1～5 对外显示为最大速度的 20%～100%。
SPEED_LEVELS = (1.0, 2.0, 3.0, 4.0, 5.0)
SPEED_PERCENT_LABELS = (20, 40, 60, 80, 100)
DEFAULT_SPEED_INDEX = 0  # 启动时为 20% 档
VIEW_NAMES = ("背面", "右侧", "正面", "左侧")
DEFAULT_VIEW_INDEX = 0
A_LONG_PRESS_EXIT_S = 2.0
RUMBLE_DURATION_MS = 1000

# O4 位置逆速度的阻尼，单位与位置雅可比一致。
O4_DAMPING_MM = 20.0
O4_SINGULAR_WARNING_MM = 8.0
WRIST_STRAIGHT_WARNING_DEG = 5.0

# 操作方向：如短脉冲实机测试方向相反，只把相应的 +1 改为 -1。
BASE_X_FORWARD_SIGN = +1.0
BASE_Y_RIGHT_SIGN = +1.0
BASE_Z_UP_SIGN = -1.0       # DH 模型 +z0 指向地面
J4_STICK_RIGHT_SIGN = +1.0
J5_RB_POSITIVE_SIGN = +1.0  # RB 增大 J5 角度，LB 减小 J5 角度
J6_DPAD_RIGHT_SIGN = +1.0

# 夹爪百分比定义。先在单舵机 Notebook 中确认 0%=张开、100%=闭合。
GRIPPER_OPEN_PERCENT = 0.0
GRIPPER_CLOSED_PERCENT = 100.0

# 遥操作时主动收紧 J1/J4/J6，避免在单圈边界附近缠绕线缆。
CONTROL_Q_MIN_DEG = np.array(kin.Q_MIN_DEG, dtype=float)
CONTROL_Q_MAX_DEG = np.array(kin.Q_MAX_DEG, dtype=float)
for index in (0, 3, 5):
    CONTROL_Q_MIN_DEG[index] = max(CONTROL_Q_MIN_DEG[index], -170.0)
    CONTROL_Q_MAX_DEG[index] = min(CONTROL_Q_MAX_DEG[index], 170.0)

# STS3215 原始速度/加速度寄存器与手柄档位联动。
# 每个内部倍率对应 150/15；显示 100% 时仍为原来的最高值 750/75。
SERVO_VELOCITY_PER_SCALE = 150
SERVO_ACCELERATION_PER_SCALE = 15
SERVO_MIN_GOAL_VELOCITY = 150
SERVO_MAX_GOAL_VELOCITY = 750
SERVO_MIN_ACCELERATION = 15
SERVO_MAX_ACCELERATION = 75

def servo_motion_profile(speed_scale):
    '''把手柄速度档位转换为 STS3215 原始速度和加速度寄存器值。'''
    if not np.isfinite(speed_scale) or not 0.0 < speed_scale <= max(SPEED_LEVELS):
        raise ValueError(f"speed_scale 必须位于 (0, {max(SPEED_LEVELS)}]。")
    velocity = int(np.floor(SERVO_VELOCITY_PER_SCALE * speed_scale + 0.5))
    acceleration = int(np.floor(SERVO_ACCELERATION_PER_SCALE * speed_scale + 0.5))
    velocity = int(np.clip(velocity, SERVO_MIN_GOAL_VELOCITY, SERVO_MAX_GOAL_VELOCITY))
    acceleration = int(np.clip(acceleration, SERVO_MIN_ACCELERATION, SERVO_MAX_ACCELERATION))
    return velocity, acceleration

def speed_percent(speed_index):
    return int(SPEED_PERCENT_LABELS[int(speed_index)])

def operator_to_base_planar(forward, right, view_index):
    '''按背面、右侧、正面、左侧视角旋转操作者的水平移动指令。'''
    rotations = (
        ((1.0, 0.0), (0.0, 1.0)),
        ((0.0, -1.0), (1.0, 0.0)),
        ((-1.0, 0.0), (0.0, -1.0)),
        ((0.0, 1.0), (-1.0, 0.0)),
    )
    matrix = np.asarray(rotations[int(view_index) % len(VIEW_NAMES)], dtype=float)
    return tuple(matrix @ np.array([forward, right], dtype=float))
MAX_TRACKING_ERROR_DEG = 15.0
MAX_TEMPERATURE_C = 70.0
STATUS_REFRESH_S = 0.25
TEMPERATURE_CHECK_S = 1.0

print("PORT =", PORT)
for name, low, high in zip(JOINTS, CONTROL_Q_MIN_DEG, CONTROL_Q_MAX_DEG):
    print(f"{name}: {low:8.3f}° ～ {high:8.3f}°")

## 4. Xbox 手柄读取

这里使用 Pygame 的 SDL GameController 接口，而不是依赖厂商驱动变化较大的原始轴编号。SDL 标准映射为：A/B、LB/RB、左右摇杆、左右扳机和方向键。运行前请先在 `lerobot` 环境安装 Pygame：

```powershell
python -m pip install pygame
```运行前请先在 `lerobot` 环境安装 Pygame：

```powershell
python -m pip install pygame
```

先用 USB 或 Xbox 无线适配器连接手柄，再运行初始化单元。首次使用必须运行 `preview_gamepad()`：松手时四个摇杆应接近 0，LT/RT 应接近 0；按键名称应与实际一致。

In [ ]:
import pygame
import pygame._sdl2.controller as sdl_controller

# SDL GameController 标准枚举值。
AXIS_LEFT_X, AXIS_LEFT_Y = 0, 1
AXIS_RIGHT_X, AXIS_RIGHT_Y = 2, 3
AXIS_TRIGGER_LEFT, AXIS_TRIGGER_RIGHT = 4, 5
BUTTON_A, BUTTON_B, BUTTON_X, BUTTON_Y = 0, 1, 2, 3
BUTTON_LB, BUTTON_RB = 9, 10
BUTTON_DPAD_UP, BUTTON_DPAD_DOWN = 11, 12
BUTTON_DPAD_LEFT, BUTTON_DPAD_RIGHT = 13, 14

def shape_stick(value, deadzone=STICK_DEADZONE, cubic_mix=STICK_CUBIC_MIX):
    '''死区外重新归一化，并用线性/三次混合曲线改善中心精细度。'''
    value = float(np.clip(value, -1.0, 1.0))
    magnitude = abs(value)
    if magnitude <= deadzone:
        return 0.0
    scaled = (magnitude - deadzone) / (1.0 - deadzone)
    shaped = (1.0 - cubic_mix) * scaled + cubic_mix * scaled**3
    return float(np.copysign(shaped, value))

def normalize_axis(raw):
    return float(np.clip(raw / 32767.0, -1.0, 1.0))

def normalize_trigger(raw):
    value = float(np.clip(raw / 32767.0, 0.0, 1.0))
    if value <= TRIGGER_DEADZONE:
        return 0.0
    return (value - TRIGGER_DEADZONE) / (1.0 - TRIGGER_DEADZONE)

def open_xbox_controller(index=0):
    pygame.init()
    sdl_controller.init()
    count = sdl_controller.get_count()
    compatible = [i for i in range(count) if sdl_controller.is_controller(i)]
    if not compatible:
        raise RuntimeError("未检测到 SDL 兼容手柄。请连接 Xbox 手柄后重试。")
    if index >= len(compatible):
        raise IndexError(f"手柄索引 {index} 无效；兼容手柄数量={len(compatible)}")
    controller = sdl_controller.Controller(compatible[index])
    print("手柄：", controller.name)
    print("SDL mapping：", controller.get_mapping())
    return controller

def read_gamepad(controller):
    pygame.event.pump()
    if not controller.attached():
        raise ConnectionError("Xbox 手柄已断开。")
    return {
        "lx": shape_stick(normalize_axis(controller.get_axis(AXIS_LEFT_X))),
        "ly": shape_stick(normalize_axis(controller.get_axis(AXIS_LEFT_Y))),
        "rx": shape_stick(normalize_axis(controller.get_axis(AXIS_RIGHT_X))),
        "ry": shape_stick(normalize_axis(controller.get_axis(AXIS_RIGHT_Y))),
        "lt": normalize_trigger(controller.get_axis(AXIS_TRIGGER_LEFT)),
        "rt": normalize_trigger(controller.get_axis(AXIS_TRIGGER_RIGHT)),
        "a": bool(controller.get_button(BUTTON_A)),
        "b": bool(controller.get_button(BUTTON_B)),
        "x": bool(controller.get_button(BUTTON_X)),
        "y": bool(controller.get_button(BUTTON_Y)),
        "lb": bool(controller.get_button(BUTTON_LB)),
        "rb": bool(controller.get_button(BUTTON_RB)),
        "dpad_left": bool(controller.get_button(BUTTON_DPAD_LEFT)),
        "dpad_right": bool(controller.get_button(BUTTON_DPAD_RIGHT)),
        "dpad_up": bool(controller.get_button(BUTTON_DPAD_UP)),
        "dpad_down": bool(controller.get_button(BUTTON_DPAD_DOWN)),
    }

def gamepad_is_centered(pad, threshold=0.08):
    analog = [pad[k] for k in ("lx", "ly", "rx", "ry", "lt", "rt")]
    digital_motion = (
        pad["dpad_left"] or pad["dpad_right"]
        or pad["dpad_up"] or pad["dpad_down"] or pad["lb"] or pad["rb"]
    )
    return max(map(abs, analog), default=0.0) <= threshold and not digital_motion

def format_gamepad_preview_html(pad, remaining_s):
    analog_rows = "".join(
        f"<tr><td>{name}</td><td>{pad[name]:+.3f}</td></tr>"
        for name in ("lx", "ly", "rx", "ry", "lt", "rt")
    )
    button_rows = " ".join(
        f"<b>{name.upper()}</b>={'按下' if pad[name] else '松开'}"
        for name in ("a", "b", "x", "y", "lb", "rb", "dpad_up", "dpad_down", "dpad_left", "dpad_right")
    )
    return f"""
    <div style="font-family:Consolas,monospace;line-height:1.45">
      <b>Xbox 输入预览</b>　剩余 {remaining_s:.1f} 秒<br>
      <span>移动摇杆、扳机并测试按键；本单元不会访问舵机。</span>
      <table style="margin-top:6px;border-collapse:collapse">{analog_rows}</table>
      <div style="margin-top:6px">{button_rows}</div>
    </div>
    """

def preview_gamepad(controller, seconds=10.0, refresh_s=0.10):
    '''只预览输入，不访问机械臂。固定更新一个面板，按 Ctrl+C 可提前结束。'''
    panel = widgets.HTML()
    display(panel)
    deadline = time.monotonic() + seconds
    try:
        while time.monotonic() < deadline:
            pad = read_gamepad(controller)
            panel.value = format_gamepad_preview_html(
                pad, max(0.0, deadline - time.monotonic())
            )
            time.sleep(refresh_s)
        panel.value += "<div style='color:#0B6E4F'><b>预览完成。</b></div>"
    except KeyboardInterrupt:
        panel.value += "<div style='color:#8A4B08'><b>预览已停止。</b></div>"

print("Xbox 读取函数已加载；尚未打开手柄，也未访问串口。")

In [ ]:
# 本单元只打开并预览手柄，不访问舵机。
input("确认 Xbox 手柄已经连接。按 Enter 打开手柄并开始 5 秒输入预览。")

if "controller" not in globals() or controller is None or not controller.attached():
    controller = open_xbox_controller(0)
preview_gamepad(controller, seconds=5.0)


## 5. 腕部中心 O4 的运动学

上方内置核心的 `forward_kinematics(q, return_all=True)` 返回 `{0}`～`{6}` 的全部齐次变换。`frames[4]` 的原点就是 `O4`。J1～J3 对 O4 的位置雅可比为：

```text
Jv,i = z(i-1) × (O4 - O(i-1)),  i=1,2,3
```

通过阻尼最小二乘把期望 O4 速度转换为 J1～J3 速度。J4、J5、J6 不参与 O4 定位，而是按手柄输入直接积分关节角。

In [ ]:
def wrist_center_position_and_jacobian(q_deg):
    '''返回 O4 位置和 J1～J3 对 O4 的 3×3 位置雅可比。'''
    q = np.asarray(q_deg, dtype=float).reshape(6)
    _, frames = kin.forward_kinematics(q, return_all=True)
    o4 = frames[4][:3, 3]
    jacobian = np.zeros((3, 3), dtype=float)
    for index in range(3):
        axis = frames[index][:3, 2]
        origin = frames[index][:3, 3]
        jacobian[:, index] = np.cross(axis, o4 - origin)
    return o4, jacobian

def wrist_center_velocity_step(q_deg, velocity_mm_s, dt, speed_scale=1.0):
    '''使用 DLS 计算移动 O4 所需的下一周期 J1～J3 目标。'''
    q = np.asarray(q_deg, dtype=float).reshape(6).copy()
    velocity = np.asarray(velocity_mm_s, dtype=float).reshape(3)
    if not np.all(np.isfinite(velocity)) or dt <= 0:
        raise ValueError("O4 速度必须为有限数，dt 必须大于 0。")
    _, jacobian = wrist_center_position_and_jacobian(q)
    singular_values = np.linalg.svd(jacobian, compute_uv=False)
    system = jacobian @ jacobian.T + O4_DAMPING_MM**2 * np.eye(3)
    q_dot_rad_s = jacobian.T @ np.linalg.solve(system, velocity)
    speed_limit = np.deg2rad(J123_SPEED_PER_LEVEL_DEG_S * speed_scale)
    q_dot_rad_s = np.clip(q_dot_rad_s, -speed_limit, speed_limit)
    q[:3] += np.rad2deg(q_dot_rad_s) * dt
    q[:3] = np.clip(q[:3], CONTROL_Q_MIN_DEG[:3], CONTROL_Q_MAX_DEG[:3])
    return q, np.rad2deg(q_dot_rad_s), singular_values

def apply_gamepad_step(
    q_target_deg, gripper_target_percent, pad, dt, speed_scale=1.0, view_index=0
):
    '''纯数学控制步；不读取手柄、不访问串口。'''
    if not 0.0 < speed_scale <= max(SPEED_LEVELS):
        raise ValueError(f"speed_scale 必须位于 (0, {max(SPEED_LEVELS)}]。")
    q_next = np.asarray(q_target_deg, dtype=float).reshape(6).copy()
    gripper_next = float(gripper_target_percent)
    details = {
        "o4_velocity_mm_s": np.zeros(3),
        "j123_velocity_deg_s": np.zeros(3),
        "o4_singular_values": np.full(3, np.nan),
        "wrist_warning": "",
    }

    # 左右摇杆都按当前视角旋转；背面视角保持原始映射。
    forward = -pad["ly"]
    right = pad["lx"]
    forward, right = operator_to_base_planar(forward, right, view_index)
    up = -pad["ry"]
    up, j4_command = operator_to_base_planar(up, pad["rx"], view_index)
    o4_velocity = speed_scale * O4_SPEED_PER_LEVEL_MM_S * np.array([
        BASE_X_FORWARD_SIGN * forward,
        BASE_Y_RIGHT_SIGN * right,
        BASE_Z_UP_SIGN * up,
    ])
    q_next, j123_velocity, singular_values = wrist_center_velocity_step(
        q_next, o4_velocity, dt, speed_scale=speed_scale
    )
    details["o4_velocity_mm_s"] = o4_velocity
    details["j123_velocity_deg_s"] = j123_velocity
    details["o4_singular_values"] = singular_values

    # 旋转后的右摇杆横向分量控制 J4；RB/LB 分别让 J5 正转/反转。
    q_next[3] += (
        J4_STICK_RIGHT_SIGN * j4_command * J4_SPEED_PER_LEVEL_DEG_S * speed_scale * dt
    )
    j5_direction = float(pad["rb"]) - float(pad["lb"])
    q_next[4] += (
        J5_RB_POSITIVE_SIGN * j5_direction * J5_SPEED_PER_LEVEL_DEG_S * speed_scale * dt
    )
    if abs(q_next[4]) < WRIST_STRAIGHT_WARNING_DEG and abs(j4_command) > 0:
        details["wrist_warning"] = "J5 接近 0°：J4 对 TCP 位置的作用很小。"

    j6_direction = float(pad["dpad_right"]) - float(pad["dpad_left"])
    q_next[5] += (
        J6_DPAD_RIGHT_SIGN * j6_direction * J6_SPEED_PER_LEVEL_DEG_S * speed_scale * dt
    )

    # RT 闭合、LT 张开；百分比方向由上方两个标定常量决定。
    close_direction = np.sign(GRIPPER_CLOSED_PERCENT - GRIPPER_OPEN_PERCENT)
    gripper_next += (
        close_direction * (pad["rt"] - pad["lt"])
        * GRIPPER_SPEED_PERCENT_S * speed_scale * dt
    )

    q_next = np.clip(q_next, CONTROL_Q_MIN_DEG, CONTROL_Q_MAX_DEG)
    gripper_next = float(np.clip(gripper_next, 0.0, 100.0))
    return q_next, gripper_next, details

def limit_wrapped_target_lead(target_deg, feedback_deg, max_lead_deg):
    '''限制连续关节目标相对反馈的最短角度差，避免目标无限领先。'''
    if max_lead_deg <= 0:
        raise ValueError("max_lead_deg 必须大于 0。")
    delta = (float(target_deg) - float(feedback_deg) + 180.0) % 360.0 - 180.0
    delta = float(np.clip(delta, -max_lead_deg, max_lead_deg))
    return float(feedback_deg) + delta

print("O4 运动学与手柄数学控制步已加载。")

## 6. 离线回归测试（不连接手柄、不访问串口）

In [ ]:
def fake_pad(**changes):
    pad = {
        "lx": 0.0, "ly": 0.0, "rx": 0.0, "ry": 0.0,
        "lt": 0.0, "rt": 0.0,
        "a": False, "b": False, "x": False, "y": False,
        "lb": False, "rb": False,
        "dpad_up": False, "dpad_down": False,
        "dpad_left": False, "dpad_right": False,
    }
    pad.update(changes)
    return pad

q_test = np.array([0.0, -20.0, 35.0, 10.0, 20.0, 0.0])
gripper_test = 50.0

# 输入全部回中时目标不得改变。
q_idle, g_idle, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(), 1.0 / LOOP_HZ
)
np.testing.assert_allclose(q_idle, q_test)
assert g_idle == gripper_test

# 平移模式：J1～J3 改变，J4～J6 保持。
q_move, _, move_info = apply_gamepad_step(
    q_test, gripper_test, fake_pad(ly=-0.7, ry=-0.5), 1.0 / LOOP_HZ
)
assert np.linalg.norm(q_move[:3] - q_test[:3]) > 0
np.testing.assert_allclose(q_move[3:], q_test[3:])

# 背面视角：右摇杆上下只移动 O4，左右只转动 J4，不依赖 LB/RB 切换。
q_right_vertical, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(ry=-0.6), 1.0 / LOOP_HZ
)
assert np.linalg.norm(q_right_vertical[:3] - q_test[:3]) > 0
np.testing.assert_allclose(q_right_vertical[3:], q_test[3:])
q_right_horizontal, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(rx=0.6), 1.0 / LOOP_HZ
)
np.testing.assert_allclose(q_right_horizontal[:3], q_test[:3])
assert q_right_horizontal[3] != q_test[3]
np.testing.assert_allclose(q_right_horizontal[4:], q_test[4:])

# 背面视角组合输入：右摇杆上下移动 O4，左右转动 J4，RB 让 J5 正转。
q_wrist, _, _ = apply_gamepad_step(
    q_test, gripper_test,
    fake_pad(rb=True, lx=0.4, rx=0.5, ry=-0.5),
    1.0 / LOOP_HZ,
)
assert np.linalg.norm(q_wrist[:3] - q_test[:3]) > 0
assert q_wrist[3] != q_test[3] and q_wrist[4] != q_test[4]

# 方向键和扳机映射。
q_aux, g_aux, _ = apply_gamepad_step(
    q_test, gripper_test,
    fake_pad(dpad_right=True, rt=1.0),
    1.0 / LOOP_HZ,
)
assert q_aux[5] > q_test[5]
np.testing.assert_allclose(q_aux[5] - q_test[5], J6_SPEED_PER_LEVEL_DEG_S / LOOP_HZ)
assert J6_SPEED_PER_LEVEL_DEG_S == J123_SPEED_PER_LEVEL_DEG_S == J4_SPEED_PER_LEVEL_DEG_S == J5_SPEED_PER_LEVEL_DEG_S
assert g_aux > gripper_test
assert limit_wrapped_target_lead(20.0, 0.0, 5.0) == 5.0
assert limit_wrapped_target_lead(-20.0, 0.0, 5.0) == -5.0
assert limit_wrapped_target_lead(-179.0, 179.0, 5.0) == 181.0

# 低速档的单周期变化必须小于高速档。
q_slow, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(ly=-0.7),
    1.0 / LOOP_HZ, speed_scale=SPEED_LEVELS[0],
)
q_fast, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(ly=-0.7),
    1.0 / LOOP_HZ, speed_scale=SPEED_LEVELS[-1],
)
assert np.linalg.norm(q_fast[:3] - q_test[:3]) > np.linalg.norm(q_slow[:3] - q_test[:3])
assert SPEED_LEVELS == (1.0, 2.0, 3.0, 4.0, 5.0)
assert SPEED_PERCENT_LABELS == (20, 40, 60, 80, 100)
assert VIEW_NAMES == ("背面", "右侧", "正面", "左侧")
expected_servo_profiles = ((150, 15), (300, 30), (450, 45), (600, 60), (750, 75))
actual_servo_profiles = tuple(servo_motion_profile(scale) for scale in SPEED_LEVELS)
assert actual_servo_profiles == expected_servo_profiles, actual_servo_profiles
assert operator_to_base_planar(1.0, 0.0, 0) == (1.0, 0.0)
assert operator_to_base_planar(1.0, 0.0, 1) == (0.0, 1.0)
assert operator_to_base_planar(1.0, 0.0, 2) == (-1.0, 0.0)
assert operator_to_base_planar(1.0, 0.0, 3) == (0.0, -1.0)
q_view_back, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(ry=-0.6), 1.0 / LOOP_HZ, view_index=2
)
np.testing.assert_allclose(q_view_back[:3] - q_test[:3], -(q_right_vertical[:3] - q_test[:3]))
q_view_right, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(ry=-0.6), 1.0 / LOOP_HZ, view_index=1
)
np.testing.assert_allclose(q_view_right[:3], q_test[:3])
assert q_view_right[3] != q_test[3]
q_j5_positive, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(rb=True), 1.0 / LOOP_HZ, speed_scale=1.0
)
q_j5_negative, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(lb=True), 1.0 / LOOP_HZ, speed_scale=1.0
)
q_j5_cancel, _, _ = apply_gamepad_step(
    q_test, gripper_test, fake_pad(rb=True, lb=True), 1.0 / LOOP_HZ, speed_scale=1.0
)
assert q_j5_positive[4] > q_test[4] and q_j5_negative[4] < q_test[4]
np.testing.assert_allclose(q_j5_cancel[4], q_test[4])

# 解析 O4 雅可比与中心差分对比。
o4, jac = wrist_center_position_and_jacobian(q_test)
eps = 1e-6
jac_num = np.zeros((3, 3))
for i in range(3):
    dq_deg = np.zeros(6)
    dq_deg[i] = np.rad2deg(eps)
    plus = wrist_center_position_and_jacobian(q_test + dq_deg)[0]
    minus = wrist_center_position_and_jacobian(q_test - dq_deg)[0]
    jac_num[:, i] = (plus - minus) / (2.0 * eps)
jac_error = float(np.max(np.abs(jac - jac_num)))
assert jac_error < 1e-4, jac_error

print("OFFLINE_TESTS_OK")
print("O4 =", np.round(o4, 3), "mm")
print("O4 Jacobian max error =", f"{jac_error:.3e}")
print("O4 singular values =", np.round(move_info["o4_singular_values"], 6))

## 7. 舵机总线与编码换算

本节只定义函数。`connect_robot()` 连接后会立即关闭全部舵机扭矩、检查位置模式并读取状态，不会主动运动。第一次 A 进入控制时，程序会先把七台舵机的当前反馈写成目标，再使能扭矩，避免跳向旧目标。

In [ ]:
from lerobot.motors import Motor, MotorNormMode
from lerobot.motors.feetech import FeetechMotorsBus, OperatingMode

ENCODER_RESOLUTION = int(params["hardware"].get("encoder_resolution", 4096))

def make_bus():
    motors = {}
    for name in MOTOR_NAMES:
        item = params["joints"][name]
        mode = MotorNormMode.RANGE_0_100 if name == "gripper" else MotorNormMode.DEGREES
        motors[name] = Motor(int(item["id"]), "sts3215", mode)
    return FeetechMotorsBus(port=PORT, motors=motors)

def raw_feedback_to_joint_deg(raw_positions):
    return np.array([
        kin.calibration_angle_deg(params["joints"][name], raw_positions[name])
        for name in JOINTS
    ], dtype=float)

def raw_to_gripper_percent(raw_value):
    item = params["joints"]["gripper"]
    low, high = int(item["soft_min_raw"]), int(item["soft_max_raw"])
    ratio = (int(raw_value) - low) / (high - low)
    if int(item.get("direction", 1)) == -1:
        ratio = 1.0 - ratio
    return float(np.clip(100.0 * ratio, 0.0, 100.0))

def gripper_percent_to_raw(percent):
    item = params["joints"]["gripper"]
    low, high = int(item["soft_min_raw"]), int(item["soft_max_raw"])
    ratio = float(np.clip(percent, 0.0, 100.0)) / 100.0
    if int(item.get("direction", 1)) == -1:
        ratio = 1.0 - ratio
    return int(round(low + ratio * (high - low)))

def targets_to_raw(q_deg, gripper_percent):
    arm_targets = kin.joint_deg_to_servo_raw(q_deg)
    if arm_targets is None:
        raise RuntimeError("内置运动学核心没有加载校准数据，不能换算舵机目标。")
    return {**arm_targets, "gripper": gripper_percent_to_raw(gripper_percent)}

bus = None
control_state = {
    "mode": "DISCONNECTED",
    "emergency_latched": False,
    "q_target_deg": None,
    "gripper_target_percent": None,
    "speed_index": DEFAULT_SPEED_INDEX,
    "view_index": DEFAULT_VIEW_INDEX,
    "last_message": "尚未连接",
}

def connect_robot():
    global bus
    if bus is not None and bus.is_connected:
        raise RuntimeError("总线已经连接。")
    bus = make_bus()
    try:
        bus.connect(handshake=True)
        bus.disable_torque(num_retry=2)
        for name in MOTOR_NAMES:
            mode = int(bus.read("Operating_Mode", name, normalize=False, num_retry=2))
            if mode != OperatingMode.POSITION.value:
                raise RuntimeError(f"{name} Operating_Mode={mode}，不是位置模式。")
        raw = bus.sync_read("Present_Position", normalize=False, num_retry=2)
        q_feedback = raw_feedback_to_joint_deg(raw)
        gripper_feedback = raw_to_gripper_percent(raw["gripper"])
        if np.any(q_feedback < CONTROL_Q_MIN_DEG) or np.any(q_feedback > CONTROL_Q_MAX_DEG):
            raise RuntimeError(
                "当前 J1/J4/J6 或其他关节位于遥操作软件范围外；"
                "保持卸力并先在支撑下处理当前位置。"
            )
        control_state.update({
            "mode": "TORQUE_OFF",
            "emergency_latched": False,
            "q_target_deg": q_feedback.copy(),
            "gripper_target_percent": gripper_feedback,
            "last_message": "已连接，当前全部卸力；按 A 可安全使能并进入控制。",
        })
        print(control_state["last_message"])
        print("反馈角：", np.round(q_feedback, 3))
        print("夹爪：", round(gripper_feedback, 2), "%")
    except Exception:
        if bus is not None and bus.is_connected:
            bus.disconnect(disable_torque=True)
        bus = None
        control_state["mode"] = "DISCONNECTED"
        raise
    return bus

print("总线函数已加载；尚未连接串口。")

## 8. 安全状态机

In [ ]:
def require_connected():
    if bus is None or not bus.is_connected:
        raise RuntimeError("舵机总线未连接。")

def read_feedback():
    require_connected()
    raw = bus.sync_read("Present_Position", normalize=False, num_retry=1)
    return raw, raw_feedback_to_joint_deg(raw), raw_to_gripper_percent(raw["gripper"])

def write_servo_motion_profile(speed_scale):
    '''把当前档位对应的速度/加速度同步写入七台舵机。'''
    require_connected()
    velocity, acceleration = servo_motion_profile(speed_scale)
    bus.sync_write(
        "Goal_Velocity", {name: velocity for name in MOTOR_NAMES},
        normalize=False, num_retry=1,
    )
    bus.sync_write(
        "Acceleration", {name: acceleration for name in MOTOR_NAMES},
        normalize=False, num_retry=1,
    )
    return velocity, acceleration

def safe_enable_and_enter_control():
    '''先同步当前位置目标，再使能全部扭矩。'''
    require_connected()
    if control_state["emergency_latched"]:
        raise RuntimeError("紧急卸力已锁存；必须显式复位后才能重新使能。")
    raw, q_feedback, gripper_feedback = read_feedback()
    if np.any(q_feedback < CONTROL_Q_MIN_DEG) or np.any(q_feedback > CONTROL_Q_MAX_DEG):
        raise RuntimeError("反馈角超出遥操作软件范围，拒绝使能。")
    bus.sync_write("Goal_Position", raw, normalize=False, num_retry=1)
    speed_scale = SPEED_LEVELS[int(control_state["speed_index"])]
    velocity, acceleration = write_servo_motion_profile(speed_scale)
    bus.enable_torque(num_retry=2)
    control_state.update({
        "mode": "CONTROL_ENABLED",
        "q_target_deg": q_feedback.copy(),
        "gripper_target_percent": gripper_feedback,
        "last_message": f"已使能；舵机速度/加速度={velocity}/{acceleration}，手柄输入可直接控制运动。",
    })

def hold_current_position(message="已退出控制并保持当前位置和力矩。"):
    require_connected()
    raw, q_feedback, gripper_feedback = read_feedback()
    bus.sync_write("Goal_Position", raw, normalize=False, num_retry=1)
    control_state.update({
        "mode": "HOLDING",
        "q_target_deg": q_feedback.copy(),
        "gripper_target_percent": gripper_feedback,
        "last_message": message,
    })

def emergency_release(message="B 键紧急卸力"):
    '''尽最大努力关闭扭矩并锁存；卸力后机械臂可能下坠。'''
    control_state.update({
        "mode": "EMERGENCY_TORQUE_OFF",
        "emergency_latched": True,
        "last_message": message,
    })
    if bus is not None and bus.is_connected:
        try:
            bus.disable_torque(num_retry=2)
        except Exception as exc:
            print("警告：卸力命令失败，请立即切断外部电源：", exc)

def reset_emergency_latch():
    '''经 Enter 确认后清除锁存；不使能扭矩。'''
    require_connected()
    input("确认机械臂已由人或支架托住、故障已排除。按 Enter 清除紧急锁存（仍保持卸力）。")
    raw, q_feedback, gripper_feedback = read_feedback()
    control_state.update({
        "mode": "TORQUE_OFF",
        "emergency_latched": False,
        "q_target_deg": q_feedback.copy(),
        "gripper_target_percent": gripper_feedback,
        "last_message": "紧急锁存已复位；仍为卸力状态，摇杆回中后按 A 才会使能。",
    })

def angular_tracking_error_deg(target, feedback):
    delta = np.asarray(target) - np.asarray(feedback)
    for index in (0, 3, 5):
        delta[index] = (delta[index] + 180.0) % 360.0 - 180.0
    return np.abs(delta)

def format_live_status_html(pad, q_feedback, gripper_feedback, details):
    '''生成紧凑的固定状态卡片；主循环只更新该卡片，不清空输出区。'''
    o4, _ = wrist_center_position_and_jacobian(q_feedback)
    speed_index = int(control_state["speed_index"])
    speed_scale = SPEED_LEVELS[speed_index]
    speed_label = speed_percent(speed_index)
    view_name = VIEW_NAMES[int(control_state["view_index"])]
    servo_velocity, servo_acceleration = servo_motion_profile(speed_scale)
    q_feedback = np.asarray(q_feedback, dtype=float)
    q_target = np.asarray(control_state["q_target_deg"], dtype=float)

    mode_styles = {
        "CONTROL_ENABLED": ("可控制", "#065F46", "#D1FAE5"),
        "HOLDING": ("保持中", "#92400E", "#FEF3C7"),
        "TORQUE_OFF": ("已卸力", "#475569", "#F1F5F9"),
        "EMERGENCY_TORQUE_OFF": ("紧急卸力", "#991B1B", "#FEE2E2"),
        "DISCONNECTED": ("未连接", "#475569", "#F1F5F9"),
    }
    mode_label, mode_color, mode_background = mode_styles.get(
        control_state["mode"], (control_state["mode"], "#334155", "#F1F5F9")
    )

    joint_headers = "".join(f"<th>{name}</th>" for name in JOINTS)
    feedback_cells = "".join(f"<td>{value:+.1f}°</td>" for value in q_feedback)
    target_cells = "".join(f"<td>{value:+.1f}°</td>" for value in q_target)
    o4_text = " / ".join(f"{value:+.1f}" for value in o4)

    warnings = []
    singular = details.get("o4_singular_values")
    singular_text = "—"
    if singular is not None and np.all(np.isfinite(singular)):
        singular_text = " / ".join(f"{value:.2f}" for value in singular)
        if singular[-1] < O4_SINGULAR_WARNING_MM:
            warnings.append("J1～J3 接近 O4 定位奇异结构，DLS 已限制速度。")
    if details.get("wrist_warning"):
        warnings.append(details["wrist_warning"])
    warning_html = "".join(
        f"<div style='margin-top:8px;padding:8px 10px;border-radius:8px;background:#FFF7ED;color:#9A3412'>⚠ {html.escape(str(item))}</div>"
        for item in warnings
    )

    return f"""
    <div style="max-width:920px;padding:14px 16px;border:1px solid #E2E8F0;border-radius:14px;
                background:#FFFFFF;box-shadow:0 4px 14px rgba(15,23,42,.07);
                color:#0F172A;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI','Microsoft YaHei',sans-serif">
      <div style="display:flex;align-items:center;justify-content:space-between;gap:12px;margin-bottom:10px">
        <div><span style="font-size:16px;font-weight:700">VEILINK 控制台</span>
             <span style="margin-left:8px;color:#64748B;font-size:12px">O4 + J4/J5 独立控制</span></div>
        <span style="padding:4px 10px;border-radius:999px;background:{mode_background};color:{mode_color};font-size:12px;font-weight:700">{html.escape(mode_label)}</span>
      </div>
      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:8px;margin-bottom:10px">
        <div style="padding:8px 10px;background:#F8FAFC;border-radius:9px"><div style="font-size:11px;color:#64748B">视角</div><b>{html.escape(view_name)}</b></div>
        <div style="padding:8px 10px;background:#F8FAFC;border-radius:9px"><div style="font-size:11px;color:#64748B">速度</div><b>{speed_label}%</b> · 第{speed_index + 1}档</div>
        <div style="padding:8px 10px;background:#F8FAFC;border-radius:9px"><div style="font-size:11px;color:#64748B">舵机 V / A</div><b>{servo_velocity} / {servo_acceleration}</b></div>
        <div style="padding:8px 10px;background:#F8FAFC;border-radius:9px"><div style="font-size:11px;color:#64748B">夹爪</div><b>{gripper_feedback:.1f}%</b></div>
      </div>
      <div style="padding:8px 10px;margin-bottom:10px;border-left:3px solid {mode_color};background:#F8FAFC;border-radius:6px;font-size:13px">{html.escape(control_state['last_message'])}</div>
      <table style="width:100%;border-collapse:collapse;text-align:center;font-family:Consolas,'Cascadia Mono',monospace;font-size:12px">
        <thead><tr style="color:#64748B"><th style="text-align:left">关节</th>{joint_headers}</tr></thead>
        <tbody>
          <tr style="border-top:1px solid #E2E8F0"><th style="padding:6px;text-align:left;color:#475569">反馈</th>{feedback_cells}</tr>
          <tr style="border-top:1px solid #F1F5F9"><th style="padding:6px;text-align:left;color:#475569">目标</th>{target_cells}</tr>
        </tbody>
      </table>
      <div style="display:flex;gap:18px;flex-wrap:wrap;margin-top:10px;padding-top:9px;border-top:1px solid #E2E8F0;font-size:12px;color:#475569">
        <span><b>O4 mm</b>　{o4_text}</span><span><b>奇异值</b>　{singular_text}</span>
      </div>
      {warning_html}
      <div style="margin-top:10px;padding-top:8px;border-top:1px solid #E2E8F0;color:#64748B;font-size:11px">
        X/Y 调速　·　↑/↓ 视角　·　短按 A 控制/保持　·　长按 A 退出　·　B 紧急卸力
      </div>
    </div>
    """

print("安全状态机已加载。")

## 9. 主控制循环

循环频率默认 30 Hz。每周期读取实际关节反馈、处理 Xbox 输入、计算下一目标并用 `sync_write` 同步发送七台舵机。主要保护：

- X 每次提高一个速度档位，Y 每次降低一个档位；档位为 20%/40%/60%/80%/100%，按住不会连续跳档；
- 换档同时联动目标轨迹和七台舵机的 `Goal_Velocity/Acceleration`：150/15、300/30、450/45、600/60、750/75；
- 方向键上/下在背面、右侧、正面、左侧四个视角间循环切换，左右摇杆指令随视角按 90°旋转；
- A 只在摇杆、扳机和方向键回中时允许进入控制；按下 A 立即震动 1 秒；
- 短按 A 切换可控制/保持状态，长按 A 2 秒退出循环并保持力矩；
- 背面视角下右摇杆上下移动 O4、左右转动 J4；其他视角按 90°旋转；RB/LB 控制 J5 正/反转；
- J1～J6 使用相同的关节角速度基准；J6 目标最多领先反馈 5°，避免慢速跟随时误差累积；
- 任一关节跟踪误差超过 15°时自动退出控制并保持，并显示具体关节；
- 每秒检查温度，超过 70°C 紧急卸力；
- 手柄断开时退出控制并保持力矩；
- 串口或计算异常时尽最大努力卸力；
- B 立即锁存卸力、震动 1 秒并结束循环。

Jupyter 中断按钮或 Ctrl+C 有时无法打断正在运行的手柄循环，因此正常退出请长按 A 2 秒。Ctrl+C 仍保留为备用路径。

In [ ]:
def run_gamepad_control(controller, duration_s=None):
    require_connected()
    previous_a = False
    previous_x = False
    previous_y = False
    previous_dpad_up = False
    previous_dpad_down = False
    a_down_since = None
    rumble_stop_at = None
    start_time = time.monotonic()
    previous_time = start_time
    next_status_time = start_time
    next_temperature_time = start_time
    details = {}
    live_panel = widgets.HTML(value=(
        "<div style='max-width:920px;padding:16px;border:1px solid #E2E8F0;"
        "border-radius:14px;background:#FFFFFF;color:#475569;font-family:Segoe UI,Microsoft YaHei,sans-serif'>"
        "<b style='color:#0F172A'>VEILINK 控制台</b>　正在读取第一帧反馈……</div>"
    ))
    display(live_panel)

    try:
        while duration_s is None or time.monotonic() - start_time < duration_s:
            cycle_start = time.monotonic()
            if rumble_stop_at is not None and cycle_start >= rumble_stop_at:
                try:
                    controller.stop_rumble()
                except Exception:
                    pass
                rumble_stop_at = None
            dt = float(np.clip(cycle_start - previous_time, 1e-4, 2.0 / LOOP_HZ))
            previous_time = cycle_start

            try:
                pad = read_gamepad(controller)
            except ConnectionError:
                if control_state["mode"] in ("CONTROL_ENABLED", "HOLDING"):
                    hold_current_position("手柄断开：已退出控制并保持力矩。")
                live_panel.value = (
                    f"<div style='color:#8A4B08'><b>{html.escape(control_state['last_message'])}</b></div>"
                )
                break

            # B 立即卸力。rumble 的 duration 参数单位为 ms，1 秒后自动停止。
            if pad["b"]:
                emergency_release("B 键紧急卸力：已锁存并结束控制循环。")
                live_panel.value = (
                    f"<div style='color:#B00020'><b>{html.escape(control_state['last_message'])}</b></div>"
                )
                try:
                    rumble_started = bool(
                        controller.rumble(1.0, 1.0, RUMBLE_DURATION_MS)
                    )
                    if rumble_started:
                        time.sleep(RUMBLE_DURATION_MS / 1000.0)
                        controller.stop_rumble()
                except Exception:
                    pass
                break

            # 方向键上/下使用按下边沿，在四个视角间循环切换。
            view_up_pressed = pad["dpad_up"] and not previous_dpad_up
            view_down_pressed = pad["dpad_down"] and not previous_dpad_down
            previous_dpad_up = pad["dpad_up"]
            previous_dpad_down = pad["dpad_down"]
            if view_up_pressed != view_down_pressed:
                view_step = 1 if view_up_pressed else -1
                control_state["view_index"] = (
                    int(control_state["view_index"]) + view_step
                ) % len(VIEW_NAMES)
                view_name = VIEW_NAMES[int(control_state["view_index"])]
                control_state["last_message"] = f"视角切换为：{view_name}。"

            # X/Y 使用按下边沿，按住按钮不会在每个周期重复换档。
            x_pressed = pad["x"] and not previous_x
            y_pressed = pad["y"] and not previous_y
            previous_x, previous_y = pad["x"], pad["y"]
            if x_pressed or y_pressed:
                old_index = int(control_state["speed_index"])
                step = 1 if x_pressed else -1
                new_index = int(np.clip(old_index + step, 0, len(SPEED_LEVELS) - 1))
                control_state["speed_index"] = new_index
                scale = SPEED_LEVELS[new_index]
                label = speed_percent(new_index)
                velocity, acceleration = write_servo_motion_profile(scale)
                if new_index == old_index:
                    boundary = "最高" if x_pressed else "最低"
                    action = f"已经是{boundary}速度档"
                else:
                    action = "速度提高" if x_pressed else "速度降低"
                control_state["last_message"] = (
                    f"{action}：第 {new_index + 1} 档（{label}%），"
                    f"舵机速度/加速度={velocity}/{acceleration}。"
                )

            # A 按下时立即震动 1 秒；短按动作在松开时执行，以便区分长按。
            a_pressed = pad["a"] and not previous_a
            a_released = not pad["a"] and previous_a
            if a_pressed:
                a_down_since = cycle_start
                try:
                    rumble_started = bool(
                        controller.rumble(0.5, 0.5, RUMBLE_DURATION_MS)
                    )
                    if rumble_started:
                        rumble_stop_at = cycle_start + RUMBLE_DURATION_MS / 1000.0
                except Exception:
                    rumble_stop_at = None

            # 长按达到阈值后无需松开，立即退出循环并保持当前位置/力矩。
            if pad["a"] and a_down_since is not None:
                if cycle_start - a_down_since >= A_LONG_PRESS_EXIT_S:
                    if control_state["mode"] in ("CONTROL_ENABLED", "HOLDING"):
                        hold_current_position("A 长按 2 秒：已退出循环并保持当前位置和力矩。")
                    else:
                        control_state["last_message"] = "A 长按 2 秒：已退出控制循环。"
                    live_panel.value = (
                        f"<div style='color:#0B6E4F'><b>{html.escape(control_state['last_message'])}</b></div>"
                    )
                    break

            if a_released and a_down_since is not None:
                held_s = cycle_start - a_down_since
                if held_s < A_LONG_PRESS_EXIT_S:
                    if control_state["emergency_latched"]:
                        control_state["last_message"] = "紧急卸力已锁存，A 不会重新使能。"
                    elif control_state["mode"] in ("TORQUE_OFF", "HOLDING"):
                        if not gamepad_is_centered(pad):
                            control_state["last_message"] = "拒绝进入控制：请松开 LB/RB 并让摇杆、扳机和方向键回中。"
                        else:
                            safe_enable_and_enter_control()
                    elif control_state["mode"] == "CONTROL_ENABLED":
                        hold_current_position()
                a_down_since = None
            previous_a = pad["a"]

            raw_feedback, q_feedback, gripper_feedback = read_feedback()

            if control_state["mode"] == "CONTROL_ENABLED":
                errors = angular_tracking_error_deg(
                    control_state["q_target_deg"], q_feedback
                )
                max_error_index = int(np.argmax(errors))
                max_error = float(errors[max_error_index])
                if max_error > MAX_TRACKING_ERROR_DEG:
                    hold_current_position(
                        f"{JOINTS[max_error_index]} 跟踪误差过大（{max_error:.1f}°）："
                        "已退出控制并保持。"
                    )

            if control_state["mode"] == "CONTROL_ENABLED":
                speed_scale = SPEED_LEVELS[int(control_state["speed_index"])]
                q_next, gripper_next, details = apply_gamepad_step(
                    control_state["q_target_deg"],
                    control_state["gripper_target_percent"],
                    pad,
                    dt,
                    speed_scale=speed_scale,
                    view_index=int(control_state["view_index"]),
                )
                q_next[5] = limit_wrapped_target_lead(
                    q_next[5], q_feedback[5], MAX_J6_TARGET_LEAD_DEG
                )
                q_next[5] = float(np.clip(
                    q_next[5], CONTROL_Q_MIN_DEG[5], CONTROL_Q_MAX_DEG[5]
                ))
                raw_targets = targets_to_raw(q_next, gripper_next)
                bus.sync_write(
                    "Goal_Position", raw_targets, normalize=False, num_retry=1
                )
                control_state["q_target_deg"] = q_next
                control_state["gripper_target_percent"] = gripper_next
            else:
                details = {}

            if cycle_start >= next_temperature_time:
                temperatures = bus.sync_read(
                    "Present_Temperature", normalize=False, num_retry=1
                )
                hottest_name = max(temperatures, key=lambda name: float(temperatures[name]))
                hottest = float(temperatures[hottest_name])
                if hottest >= MAX_TEMPERATURE_C:
                    emergency_release(
                        f"{hottest_name} 温度 {hottest:.1f}°C 超限，已紧急卸力。"
                    )
                    live_panel.value = (
                        f"<div style='color:#B00020'><b>{html.escape(control_state['last_message'])}</b></div>"
                    )
                    break
                next_temperature_time = cycle_start + TEMPERATURE_CHECK_S

            if cycle_start >= next_status_time:
                live_panel.value = format_live_status_html(
                    pad, q_feedback, gripper_feedback, details
                )
                next_status_time = cycle_start + STATUS_REFRESH_S

            remaining = 1.0 / LOOP_HZ - (time.monotonic() - cycle_start)
            if remaining > 0:
                time.sleep(remaining)
        else:
            # 有限时长正常结束时也要退出控制并保持，不能让循环静默结束。
            if control_state["mode"] in ("CONTROL_ENABLED", "HOLDING"):
                hold_current_position("设定控制时长结束：已保持当前位置和力矩。")
            else:
                control_state["last_message"] = "设定控制时长结束。"
            live_panel.value = (
                f"<div style='color:#0B6E4F'><b>{html.escape(control_state['last_message'])}</b></div>"
            )

    except KeyboardInterrupt:
        if bus is not None and bus.is_connected and control_state["mode"] != "EMERGENCY_TORQUE_OFF":
            hold_current_position("Ctrl+C：已退出控制并保持力矩；如需卸力请运行关闭单元。")
        live_panel.value = (
            f"<div style='color:#8A4B08'><b>{html.escape(control_state['last_message'])}</b></div>"
        )
    except Exception as exc:
        emergency_release(f"控制循环异常：{type(exc).__name__}: {exc}")
        print(control_state["last_message"])
        print("已尝试紧急卸力；请同时切断外部电源并检查机械臂。")
        raise

print("主控制循环已加载，尚未运行。")


## 10. 按 Enter 连接硬件

执行前确认：

- 离线测试已经显示 `OFFLINE_TESTS_OK`；
- 手柄预览的各轴和按键完全正确；
- 七台电机均已接通，机械臂固定且有人准备托住承重关节；
- 串口号正确，其他控制程序已经关闭；
- 实体断电方式触手可及。

运行下一单元后，程序会等待确认；只有按下 Enter 才会打开串口。连接后全部舵机仍保持卸力，第一次按 A 才会同步当前位置并使能。

In [ ]:
input(
    "确认离线测试、手柄预览、机械支撑、串口和实体断电方式均已检查。"
    "按 Enter 连接机械臂；如不准备连接，请中止本单元。"
)

if "controller" not in globals() or controller is None or not controller.attached():
    controller = open_xbox_controller(0)
pad_now = read_gamepad(controller)
if not gamepad_is_centered(pad_now):
    raise RuntimeError("连接前请松开 LB/RB，并让摇杆、扳机和方向键全部回中。")
connect_robot()


## 11. 按 Enter 启动手柄控制

第一次实机验证可以设置有限时长；确认方向、限速和停止逻辑全部正确后，可把 `CONTROL_DURATION_S` 改为 `None` 持续运行。

运行下一单元后还会等待一次 Enter 确认。控制中：

- X 提高速度档位，Y 降低速度档位；共有20%～100%五档，100% 对应第 5 档的最高设定速度；
- 方向键上/下切换背面、右侧、正面、左侧视角；状态面板会显示当前视角；
- 按下 A 立即震动 1 秒；短按 A 切换可控制/保持状态；
- 长按 A 2 秒退出循环并保持当前位置和力矩；
- 右摇杆上下控制 O4 上下、左右控制 J4；RB/LB 分别控制 J5 正转/反转；
- B：立即卸力、震动 1 秒并退出；机械臂可能受重力下落。

In [ ]:
CONTROL_DURATION_S = None

require_connected()
if controller is None or not controller.attached():
    raise RuntimeError("Xbox 手柄未连接。请先运行手柄预览单元。")
input(
    f"确认机械臂周围无人、摇杆回中并可随时按 B。"
    f"按 Enter 启动控制循环（时长：{CONTROL_DURATION_S} 秒）。"
)
pad_now = read_gamepad(controller)
if not gamepad_is_centered(pad_now):
    raise RuntimeError("启动前请松开 LB/RB，并让摇杆、扳机和方向键全部回中。")
run_gamepad_control(controller, duration_s=CONTROL_DURATION_S)


## 12. 紧急锁存复位

B 键卸力后，A 不会直接重新使能。只有在机械臂已经由人或支架托住、故障原因已经排除后，才能运行：

```python
reset_emergency_latch()
```

函数内部会等待 Enter 确认。它只清除锁存，仍保持卸力；随后重新启动控制循环，摇杆回中后按 A 才会再次执行“当前位置写入目标 → 使能扭矩”。

## 13. 按 Enter 卸力并断开串口

每次结束 Notebook 都应运行最后一个单元。运行后程序会等待 Enter；按下前必须由人或支架托住机械臂。关闭 kernel 不能替代这个步骤；如果程序状态不确定，直接切断舵机外部电源。

In [ ]:
if bus is None or not bus.is_connected:
    print("舵机总线当前未连接。")
else:
    input("确认机械臂已由人或支架托住。按 Enter 卸力并断开串口。")
    try:
        bus.disable_torque(num_retry=2)
    finally:
        bus.disconnect(disable_torque=True)
        control_state.update({
            "mode": "DISCONNECTED",
            "emergency_latched": False,
            "last_message": "全部舵机已卸力，串口已断开。",
        })
    if controller is not None:
        try:
            controller.quit()
        except Exception:
            pass
    pygame.quit()
    print(control_state["last_message"])


## 14. 首次实机验证顺序

1. 不连接机械臂，完成手柄预览和离线回归测试。
2. 把机械臂置于远离限位的稳定姿态，托住承重关节后连接总线。
3. 确认所有运动输入回中后按 A 使能，机械臂应只保持当前位置。
4. 分别用短脉冲测试左摇杆前后/左右、右摇杆上下移动 O4，以及右摇杆左右转动 J4。
5. 在 J5 远离限位的状态下短按 RB/LB，确认分别使 J5 正转/反转，每次不超过 2°～5°。
6. 测试方向键左右控制 J6；再用方向键上/下依次切换四个视角，确认水平移动方向符合当前站位。
7. 测试 LT/RT 夹爪方向，确认 0%=张开、100%=闭合；若相反，交换上方两个百分比常量。
8. 测试 X/Y 五档换速和边界限制，确认状态面板显示 20%～100%，且按住按钮不会连续跳档。
9. 短按 A，确认退出控制后仍稳定保持，所有运动输入不再改变目标。
10. 重新进入控制后长按 A 2 秒，确认循环退出且机械臂继续保持力矩。
11. 在有人托住机械臂的前提下测试 B，确认只震动约 1 秒、进入锁存卸力且 A 不能直接恢复。
12. 所有单项都通过后，才逐步延长控制时间和提高速度。